# Tutorial 2 — First steps with IGM

*Session 2, hands-on (≈ 1 hour), then continue with your mini-project.*

Run this notebook in your RenkuLab session, from the folder `session2/tutorial-2/`.

**How we work with IGM** (the same in your mini-project):

| What | Where |
|---|---|
| **Set** the parameters | YAML files in `experiment/` (double-click to open them in the text editor) |
| **Run** IGM | a **Terminal** (*File → New → Terminal*) |
| **Look** at the results | this **notebook**, which only reads the run folders `outputs/<run>/` |

Put the notebook and the terminal side by side (drag a tab to the right half of the screen).

**Goals**
- Understand how an IGM run is set up (parameter file, inputs/processes/outputs)
- Run IGM on Aletsch Glacier with a simple ELA-based SMB (**Run A**)
- Run IGM on **any glacier** from its RGI ID, with a climate-driven SMB (**Run B**) → try it with **your** glacier
- (Optional) Plug your own SMB code into IGM (**Run C**)

IGM documentation: https://igm-model.org — the runs below are adapted from the official examples https://github.com/instructed-glacier-model/igm-examples (`aletsch` and `quick-demo`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../utils"))   # course plotting helpers
from helper import *
!pip show igm-model | head -2
!nvidia-smi -L 2>/dev/null || echo "No GPU detected: IGM runs on CPU (slower, but works)"


## 1. Folder structure

An IGM "experiment folder" always looks like this:

```
tutorial-2/
├── data/              # input data (here: input.nc for Aletsch)
├── experiment/        # parameter files: one YAML file = one experiment
│   ├── common/        # shared pieces of configuration (here: the ice-flow emulator)
│   ├── params_A_aletsch.yaml
│   ├── params_B_any_glacier.yaml
│   └── params_C_custom_smb.yaml
└── user/              # your own Python modules (optional)
```

IGM must be run **from this folder** (the one that contains `experiment/`).

👉 **Open a terminal now** (*File → New → Terminal*), put it next to this notebook, and go to this folder:

```bash
cd ~/work/glamod/session2/tutorial-2
```


In [ ]:
!tree -L 3 --noreport -I outputs

## 2. Run A — Aletsch Glacier with a simple SMB

### 2.1 The parameter file

Let us look at `experiment/params_A_aletsch.yaml`. It (i) selects the modules — **inputs** (where the data comes from), **processes** (what is computed at each time step, in this order), **outputs** (what is written) — and (ii) sets the parameters that differ from IGM's defaults.

In [ ]:
!cat experiment/params_A_aletsch.yaml

IGM is organized **module-wise**:

![](https://igm-model.org/latest/modules/flowchat-module.png)

### 2.2 The input data

The `local` input module reads `data/input.nc`, a NetCDF file with gridded fields at 100 m resolution: surface elevation `usurf`, ice thickness `thk`, ice mask, and observations (surface speeds `uvelsurfobs`, `vvelsurfobs`, ...).

In [ ]:
import xarray as xr
ds = xr.open_dataset("data/input.nc")
print(ds)
plot_input_data("data/input.nc")

### 2.3 The surface mass balance

With `smb: method: simple`, the SMB is a piecewise linear function of elevation $z$:

$$
SMB(z) = \begin{cases}
\beta_\mathrm{abl} \, (z - z_\mathrm{ELA}) & \text{if } z < z_\mathrm{ELA} \quad \text{(ablation area)} \\
\min\left(\beta_\mathrm{acc} \, (z - z_\mathrm{ELA}),\ b_\mathrm{max}\right) & \text{if } z \geq z_\mathrm{ELA} \quad \text{(accumulation area)}
\end{cases}
$$

In the parameter file, each line of the array gives (`time`, `gradabl` $=\beta_\mathrm{abl}$, `gradacc` $=\beta_\mathrm{acc}$, `ela` $= z_\mathrm{ELA}$, `accmax` $=b_\mathrm{max}$); values are linearly interpolated in time. Here the ELA rises from 2800 m (1900) to 2900 m (2000).

In [ ]:
plot_smb_simple(ela=2800, gradabl=0.009, gradacc=0.005, accmax=2.0)

### 2.4 Run IGM

For this **first run only**, we launch IGM from the notebook: the `!` sends the command to the shell. `hydra.run.dir` gives a **name** to the run folder (otherwise IGM uses `outputs/<date>/<time>/`). This takes about 1–2 minutes on a CPU, less on a GPU.

*From now on, we will type the IGM commands in the terminal: a run in the terminal keeps going if you close the browser tab, you see its progress live, and the notebook stays free for plotting.*

In [ ]:
!igm_run +experiment=params_A_aletsch hydra.run.dir=outputs/A_reference

### 2.5 Explore the results

Each run folder (here `outputs/A_reference/`) contains:
- `output.nc`: maps of thickness, surface, velocities, SMB, ... every `save` years
- `output_ts.nc`: ice volume and area through time
- `dashboard_XXXX.png`: quick-look images (double-click them in the file browser)
- `.hydra/config.yaml`: the **complete** configuration used (all parameters, including defaults)

In [ ]:
run_A = "outputs/A_reference"
show_dashboard(run_A)

In [ ]:
analyze_time_series(os.path.join(run_A, "output_ts.nc"))
animate_glacier_evolution(os.path.join(run_A, "output.nc"), variable="thk")

In [ ]:
# Modelled vs observed surface speed (at the start of the run)
compare_modeled_observed_velocity("data/input.nc", os.path.join(run_A, "output.nc"), year=1900)

✅ **Your turn!** Now in the **terminal** 👈

You can change parameters **without editing the file**, directly on the command line, and send the results to a named folder with `hydra.run.dir`. For example, a run with a 200 m higher ELA all along:

```bash
igm_run +experiment=params_A_aletsch \
   'processes.smb.simple.array=[["time","gradabl","gradacc","ela","accmax"],[1900,0.009,0.005,3000,2.0],[2100,0.009,0.005,3100,2.0]]' \
   hydra.run.dir=outputs/A_ela_plus200
```

1. Run it in the terminal, then compare with the reference run in the cell below.
2. Make the bed more slippery: `processes.iceflow.physics.sliding.tau_ref=0.15` (default in this file: 0.325 MPa), into `outputs/A_tau015`. How do the speeds and the glacier change? Compare with the observed speeds (`compare_modeled_observed_velocity`).
3. Make the ice softer: `processes.iceflow.physics.viscosity.arrhenius=150` (default: 78 MPa⁻³ a⁻¹), into `outputs/A_arrh150`.

*Tip: for longer series of experiments, copy the YAML file (file browser: right-click → Duplicate, then rename it, e.g. `params_A_ela.yaml`) and edit the copy: it documents what you did. Run it with `igm_run +experiment=params_A_ela hydra.run.dir=outputs/A_ela`.*

In [ ]:
# Plot your runs (add the folders of the runs you made in the terminal)
compare_runs([run_A, "outputs/A_ela_plus200"], ["reference", "ELA +200 m"])

## 3. Run B — Any glacier, with a climate-driven SMB

Run A had two limitations: the data was prepared in advance, and the SMB did not depend on climate. Run B (`params_B_any_glacier.yaml`, adapted from the IGM example `quick-demo`) fixes both:

1. **`oggm_shop`** downloads all the data for any glacier given its **RGI v7 ID**: topography, ice thickness, outline, observed velocities, monthly climate since 1901, and calibrated SMB parameters (via [OGGM](https://oggm.org)). The data is stored in `data/input_<RGI_ID>.nc` and `data/<RGI_ID>/`, and **re-used** in the next runs.
2. **`climate`** (`method: oggm`) provides monthly temperature and precipitation:
   - **within the climate record (1901–2019)**, the observed climate of each year is used, as is;
   - **outside the record** (before 1901, after 2019), each year is drawn randomly from the reference period 1960–1990, then shifted by a temperature offset `delta_temp` and scaled by a precipitation factor `prec_scal`, interpolated in time from:

```yaml
clim_trend_array:
  - ["time", "delta_temp", "prec_scal"]
  - [ 2020,     1.3,          1.0]     # ≈ present-day climate (2000-2019 was ~1.3°C warmer than 1960-1990)
  - [ 2100,     4.0,          1.0]     # +4°C vs 1960-1990 in 2100, same precipitation
```
   ⚠️ `delta_temp` is relative to **1960–1990**, not to today. `plot_climate` prints the present-day offset for your glacier.
3. **`smb`** (`method: oggm`) is a monthly temperature-index model: accumulation = solid precipitation, melt = melt factor × positive temperatures. `melt_enhancer` scales the melt.

In [ ]:
!cat experiment/params_B_any_glacier.yaml

✅ **Your turn!** In the **terminal**, run Run B with the RGI ID of **your glacier** (from `session1/1d-choose-your-glacier.md`; here Findelengletscher). The first run downloads the data (≈ 1 min):

```bash
igm_run +experiment=params_B_any_glacier inputs.oggm_shop.RGI_ID=RGI2000-v7.0-G-11-01189 \
   hydra.run.dir=outputs/B_RGI2000-v7.0-G-11-01189
```

When it is finished, put the same RGI ID below and run the cells to look at the results.

In [ ]:
RGI_ID = "RGI2000-v7.0-G-11-01189"   # <- the glacier of your run (here: Findelengletscher)
run_B = f"outputs/B_{RGI_ID}"

In [ ]:
plot_input_data(f"data/input_{RGI_ID}.nc")
plot_climate(RGI_ID)
show_dashboard(run_B)

In [ ]:
analyze_time_series(os.path.join(run_B, "output_ts.nc"))
compare_modeled_observed_velocity(f"data/input_{RGI_ID}.nc", os.path.join(run_B, "output.nc"), year=2020)

✅ **Your turn!**
1. In the terminal, run a weaker warming scenario (e.g. +2°C vs 1960–1990 in 2100) into another folder, then compare the two scenarios with `compare_runs` below. *Hint:* copy the YAML file and change the 2100 row of `clim_trend_array`, or override it on the command line:
   ```bash
   igm_run +experiment=params_B_any_glacier inputs.oggm_shop.RGI_ID=RGI2000-v7.0-G-11-01189 \
      'processes.climate.oggm.clim_trend_array=[[time,delta_temp,prec_scal],[2020,1.3,1.0],[2100,2.0,1.0]]' \
      hydra.run.dir=outputs/B_plus2
   ```
2. Is the modelled speed in 2020 close to the observed one? Which parameter would you change to improve this?

**This is the starting point of your mini-project** → `project/project.md` and `project/my-glacier/`.

In [ ]:
# compare_runs([run_B, "outputs/B_plus2"], ["+4°C", "+2°C"])

## 4. Run C (optional) — Your own SMB module

IGM can be extended with your own Python modules. `params_C_custom_smb.yaml` replaces the built-in `smb` by a user module `mysmb` (file `user/code/processes/mysmb.py`), in which the ELA oscillates with a period of 200 years. Every module has three functions: `initialize(cfg, state)` (called once), `update(cfg, state)` (called at each time step) and `finalize(cfg, state)`. They read and modify the shared `state` (e.g. `state.usurf`, `state.t`, `state.smb`).

In [ ]:
!cat user/code/processes/mysmb.py

Run it in the **terminal** (≈ 4 min on a CPU):

```bash
igm_run +experiment=params_C_custom_smb hydra.run.dir=outputs/C_custom_smb
```
then plot:

In [ ]:
analyze_time_series("outputs/C_custom_smb/output_ts.nc")

Compare the timing of the ELA oscillation with the timing of the volume response: what is the response time of Aletsch Glacier? Compare with Tutorial 1.

## Going further

- IGM examples: https://github.com/instructed-glacier-model/igm-examples (e.g. `aletsch`: realistic climate, particle tracking for moraines, calibration with Optuna)
- All modules and parameters: https://igm-model.org